# Linear recovery and identifiability

## Checkpoint A — controlled noiseless OLS

This executable checkpoint uses one seeded generated world, not a corpus survey. The production symbolic graph generates mixed nontrivial treatment trajectories and the observed outcome. The fit uses the true mechanism basis: each treatment column is carried over first and saturated second with its parameter-only generated scale.

Assumptions for this checkpoint:

- all three direct treatment inputs and both observed controls are active;
- geometric, Weibull, and geometric carryover are paired with Hill, Michaelis–Menten, and root saturation;
- the latent node is structurally disconnected, with no treatment mediation or treatment interactions;
- the intrinsic baseline is exactly constant at 5, the floor is disabled, and outcome noise is exactly zero;
- four generated warmup rows are retained as initial carryover history and excluded from the fit;
- OLS is run on the raw `[mechanism features, controls, intercept]` design, without centering or rescaling.

This is a positive identifiability construction. It does not estimate the frequency of rank deficiency or address noisy recovery (Checkpoint B) or an intentionally non-identifiable likelihood (Checkpoint C).


In [1]:
# Keep this report independently runnable from docs/examples/.  This fixture
# intentionally mirrors tests/test_linear_recovery.py: the construction and
# numerical assertions below must remain aligned with the focused contract.
from dataclasses import dataclass

import numpy as np
import pytensor
import pytensor.tensor as pt

from pymc_generator import mechanisms
from pymc_generator.sampler import CARRYOVER_FAMILY_KEYS, SATURATION_FAMILY_KEYS
from pymc_generator.symbolic_graph import build_symbolic_graph

SEED = 20261005
N_TIME_STEPS = 64
WARMUP = 4
L_MAX = 4
N_TREATMENTS = 3
N_COVARIATES = 2
N_LATENT = 1


@dataclass(frozen=True)
class ControlledWorld:
    outputs: dict[str, np.ndarray]
    params: dict[str, object]
    graph: dict[str, np.ndarray]


def _rw(mean, std, *, positive_only: bool) -> dict[str, object]:
    mean_array = np.asarray(mean, dtype=float)
    return {
        "mean": mean_array,
        "std": np.asarray(std, dtype=float),
        "smoothness": np.zeros(mean_array.size),
        "positive_only": positive_only,
        "rw_smoothness_max_weeks": 26,
    }


def _controlled_world() -> ControlledWorld:
    n_full = N_TIME_STEPS + WARMUP
    rng = np.random.default_rng(SEED)
    eps = {
        "eps_d": np.zeros((n_full, N_LATENT)),
        "eps_z": rng.normal(size=(n_full, N_COVARIATES)),
        "eps_c": rng.normal(size=(n_full, N_TREATMENTS)),
        "eps_b": np.zeros(n_full),
        "eps_y": np.zeros(n_full),
        "eps_c_hf": np.zeros((n_full, N_TREATMENTS)),
        "eps_c_pulse": np.zeros((n_full, N_TREATMENTS)),
    }
    params: dict[str, object] = {
        "l_max": L_MAX,
        "carryover_family": np.array(
            [CARRYOVER_FAMILY_KEYS.index(name) for name in ("geometric", "weibull", "geometric")]
        ),
        "carryover_alpha": np.array([0.42, 0.61, 0.73]),
        "weibull_lam": np.array([2.4, 3.1, 3.7]),
        "weibull_k": np.array([1.9, 2.3, 2.7]),
        "sat_family": np.array(
            [SATURATION_FAMILY_KEYS.index(name) for name in ("hill", "michaelis_menten", "root")]
        ),
        "hill_slope": np.array([1.8, 2.0, 2.2]),
        "hill_kappa_mult": np.array([0.85, 1.0, 1.15]),
        "logistic_lam": np.ones(N_TREATMENTS),
        "mm_kappa_mult": np.array([0.9, 1.25, 1.0]),
        "tanh_c": np.ones(N_TREATMENTS),
        "root_alpha": np.array([0.65, 0.72, 0.83]),
        "rw_d": _rw([0.0], [0.0], positive_only=False),
        "rw_z": _rw([0.25, -0.45], [0.55, 0.45], positive_only=False),
        "rw_c": _rw([0.45, 0.95, 1.35], [0.55, 0.65, 0.5], positive_only=True),
        "rw_b": _rw([5.0], [0.0], positive_only=False),
        "rw_y": _rw([0.0], [0.0], positive_only=False),
        "u_dz": np.zeros((N_LATENT, N_COVARIATES)),
        "gamma_zz": np.zeros((N_COVARIATES, N_COVARIATES)),
        "w_dc": np.zeros((N_LATENT, N_TREATMENTS)),
        "v_zc": np.zeros((N_COVARIATES, N_TREATMENTS)),
        "alpha_cc": np.zeros((N_TREATMENTS, N_TREATMENTS)),
        "delta_dy": np.zeros(N_LATENT),
        "rho_zy": np.array([0.35, -0.28]),
        "beta": np.array([1.15, 0.82, 1.08]),
        "hf_sigma": np.zeros(N_TREATMENTS),
        "pulse_amp": np.zeros(N_TREATMENTS),
        "pulse_prob": np.zeros(N_TREATMENTS),
        "covariate_hf_sigma": np.zeros(N_COVARIATES),
        "covariate_pulse_amp": np.zeros(N_COVARIATES),
        "covariate_pulse_prob": np.zeros(N_COVARIATES),
        "baseline_floor": None,
    }
    graph = {
        "g_cy": np.ones(N_TREATMENTS, dtype=int),
        "g_dc": np.zeros((N_LATENT, N_TREATMENTS), dtype=int),
        "g_dz": np.zeros((N_LATENT, N_COVARIATES), dtype=int),
        "g_dy": np.zeros(N_LATENT, dtype=int),
        "g_zy": np.ones(N_COVARIATES, dtype=int),
        "g_zc": np.zeros((N_COVARIATES, N_TREATMENTS), dtype=int),
        "g_cc": np.zeros((N_TREATMENTS, N_TREATMENTS), dtype=int),
        "g_zz": np.zeros((N_COVARIATES, N_COVARIATES), dtype=int),
    }
    symbolic = build_symbolic_graph(
        graph,
        params,
        n_full,
        N_TREATMENTS,
        N_COVARIATES,
        N_LATENT,
        burn_in=0,
        eps=eps,
    )["outputs"]
    evaluate = pytensor.function([], list(symbolic.values()))
    outputs = {name: np.asarray(value) for name, value in zip(symbolic, evaluate(), strict=True)}
    return ControlledWorld(outputs=outputs, params=params, graph=graph)


def _evaluate(expression: pt.TensorVariable) -> np.ndarray:
    return np.asarray(pytensor.function([], expression)())


def _mechanism_features(
    treatments: np.ndarray, scales: np.ndarray, params: dict[str, object]
) -> np.ndarray:
    features = []
    carryover_family = np.asarray(params["carryover_family"])
    saturation_family = np.asarray(params["sat_family"])
    for k in range(N_TREATMENTS):
        column = pt.as_tensor_variable(treatments[:, k, None])
        carry_id = int(carryover_family[k])
        if carry_id == CARRYOVER_FAMILY_KEYS.index("none"):
            carried = column[:, 0]
        elif carry_id == CARRYOVER_FAMILY_KEYS.index("geometric"):
            carried = mechanisms.apply_geometric_carryover(
                column, np.asarray(params["carryover_alpha"])[k], L_MAX
            )[:, 0]
        else:
            carried = mechanisms.apply_weibull_pdf_carryover(
                column,
                np.asarray(params["weibull_lam"])[k],
                np.asarray(params["weibull_k"])[k],
                L_MAX,
            )[:, 0]
        carried_values = _evaluate(carried)
        family = SATURATION_FAMILY_KEYS[int(saturation_family[k])]
        if family == "linear":
            saturated = pt.as_tensor_variable(carried_values / scales[k])
        elif family == "hill":
            saturated = mechanisms.SATURATION_FAMILIES[family](
                pt.as_tensor_variable(carried_values),
                scales[k],
                slope=np.asarray(params["hill_slope"])[k],
                kappa_mult=np.asarray(params["hill_kappa_mult"])[k],
            )
        elif family == "logistic":
            saturated = mechanisms.SATURATION_FAMILIES[family](
                pt.as_tensor_variable(carried_values),
                scales[k],
                lam=np.asarray(params["logistic_lam"])[k],
            )
        elif family == "michaelis_menten":
            saturated = mechanisms.SATURATION_FAMILIES[family](
                pt.as_tensor_variable(carried_values),
                scales[k],
                kappa_mult=np.asarray(params["mm_kappa_mult"])[k],
            )
        elif family == "tanh":
            saturated = mechanisms.SATURATION_FAMILIES[family](
                pt.as_tensor_variable(carried_values),
                scales[k],
                c=np.asarray(params["tanh_c"])[k],
            )
        else:
            saturated = mechanisms.SATURATION_FAMILIES[family](
                pt.as_tensor_variable(carried_values),
                scales[k],
                alpha=np.asarray(params["root_alpha"])[k],
            )
        features.append(_evaluate(saturated))
    return np.column_stack(features)


def _fit(world: ControlledWorld) -> tuple[np.ndarray, np.ndarray, int, float]:
    outputs = world.outputs
    params = world.params
    features_full = _mechanism_features(outputs["treatments"], outputs["saturation_scale"], params)
    window = slice(WARMUP, None)
    design = np.column_stack(
        [features_full[window], outputs["covariates"][window], np.ones(N_TIME_STEPS)]
    )
    coefficient, _residuals, rank, _singular_values = np.linalg.lstsq(
        design, outputs["outcome"][window], rcond=None
    )
    truth = np.concatenate([np.asarray(params["beta"]), np.asarray(params["rho_zy"]), [5.0]])
    return coefficient, truth, int(rank), float(np.linalg.cond(design))

world = _controlled_world()
coefficient, truth, rank, condition = _fit(world)
features_full = _mechanism_features(
    world.outputs["treatments"], world.outputs["saturation_scale"], world.params
)
window = slice(WARMUP, None)
design = np.column_stack([
    features_full[window],
    world.outputs["covariates"][window],
    np.ones(N_TIME_STEPS),
])
error = coefficient - truth
print(f"seed={SEED}, generated_rows={N_TIME_STEPS + WARMUP}, warmup={WARMUP}")
print(f"raw_design_shape={design.shape}, rank={rank}/{design.shape[1]}")
print(f"raw_condition={condition:.12g}, max_abs_error={np.max(np.abs(error)):.12g}")
print("column        truth                estimate             error")
for name, target, estimate, delta in zip(
    ["treatment_0", "treatment_1", "treatment_2", "control_0", "control_1", "intercept"],
    truth,
    coefficient,
    error,
    strict=True,
):
    print(f"{name:12s} {target: .16g} {estimate: .16g} {delta: .3g}")


/home/teemu/pymc-labs/prior-generator/.worktrees/issue-30-identifiable-linear-recovery/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


seed=20261005, generated_rows=68, warmup=4
raw_design_shape=(64, 6), rank=6/6
raw_condition=71.8975337829, max_abs_error=7.43849426499e-15
column        truth                estimate             error
treatment_0   1.15  1.15  0
treatment_1   0.82  0.8200000000000074  7.44e-15
treatment_2   1.08  1.080000000000001  4.44e-16
control_0     0.35  0.3500000000000037  3.72e-15
control_1    -0.28 -0.2800000000000023 -2.28e-15
intercept     5  4.999999999999998 -1.78e-15


### Independent feature and history checks

The generated direct contributions agree with the independently re-applied mechanism features multiplied by the generator's true treatment coefficients. Recomputing carryover from only the post-warmup window changes the first retained feature, demonstrating why the generated history is part of the basis.


In [2]:
np.testing.assert_allclose(
    world.outputs["contributions_observed"][window],
    features_full[window] * np.asarray(world.params["beta"]),
    rtol=2e-12,
    atol=2e-12,
)
reset_features = _mechanism_features(
    world.outputs["treatments"][window], world.outputs["saturation_scale"], world.params
)
history_difference = float(np.max(np.abs(reset_features[0] - features_full[WARMUP])))
np.testing.assert_array_equal(world.outputs["outcome_noise"], 0.0)
np.testing.assert_array_equal(world.outputs["baseline_intrinsic"], 5.0)
np.testing.assert_allclose(
    world.outputs["baseline"],
    world.outputs["baseline_intrinsic"] + world.outputs["covariate_contribution"].sum(axis=1),
    rtol=0.0,
    atol=2e-15,
)
assert rank == design.shape[1]
assert condition < 100.0
np.testing.assert_allclose(coefficient, truth, rtol=2e-12, atol=2e-12)
print(f"max feature contribution discrepancy={np.max(np.abs(world.outputs['contributions_observed'][window] - features_full[window] * world.params['beta'])):.3g}")
print(f"reset-at-window history discrepancy={history_difference:.6g}")


max feature contribution discrepancy=1.11e-16
reset-at-window history discrepancy=0.585202


### Result and limitation

The raw design is full rank and its condition number is reported above in the original feature units. The coefficient errors are at floating-point roundoff. The condition number is a property of this deliberately controlled trajectory, not a claim about the prior or general generated worlds. A future checkpoint must separately test noise and construct an explicit non-identifiable likelihood; neither is included here.
